# Emotion Recognition from Speech using LSTM
**Environment:** Google Colab  
**Dataset:** RAVDESS Speech Audio (8 emotions)  
**Features:** 40 MFCC coefficients over time  
**Model:** LSTM  
**Interface:** Gradio audio upload + emotion probabilities

> This is an educational prototype. The predicted label and probability are model outputs, not reliable proof of a person's actual emotional state. Performance may be lower on real-world voices, languages, microphones, accents, and recording conditions not represented in RAVDESS.

## Step 1 — Enable GPU (recommended)
In Colab, select **Runtime → Change runtime type → T4 GPU** (if available). CPU also works but may train more slowly.

In [ ]:
# Install project dependencies
!pip -q install librosa soundfile scikit-learn matplotlib seaborn gradio requests

## Step 2 — Download and extract the RAVDESS speech dataset
This downloads the smaller 16 kHz speech-only archive from Zenodo. The archive is about 78 MB. Official record: https://zenodo.org/records/11063852

In [ ]:
from pathlib import Path
import zipfile, requests, os

PROJECT_DIR = Path("/content/Emotion_Recognition_LSTM")
DATA_DIR = PROJECT_DIR / "data"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

for folder in [DATA_DIR, MODELS_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DATA_DIR / "Audio_Speech_Actors_01-24_16k.zip"
DATASET_URL = "https://zenodo.org/records/11063852/files/Audio_Speech_Actors_01-24_16k.zip?download=1"

if not ZIP_PATH.exists():
    response = requests.get(DATASET_URL, stream=True, timeout=120)
    response.raise_for_status()
    with open(ZIP_PATH, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Downloaded:", ZIP_PATH, "Size (MB):", round(ZIP_PATH.stat().st_size / 1e6, 1))

extract_dir = DATA_DIR / "ravdess_extracted"
if not extract_dir.exists() or not any(extract_dir.rglob("*.wav")):
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(extract_dir)

wav_files = list(extract_dir.rglob("*.wav"))
print("WAV files found:", len(wav_files))
print("Example:", wav_files[0] if wav_files else "No WAV files found")
if not wav_files:
    raise RuntimeError("No WAV files found. Check the ZIP download and extraction.")

## Step 3 — Imports and label mapping
RAVDESS filenames have seven numeric fields. The third field is the emotion code; the last field is the actor ID. We use actor-separated train/validation/test sets to reduce the risk of the same speaker appearing in both training and test data.

In [ ]:
import json, random, warnings
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

EMOTION_MAP = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}
EMOTIONS = [EMOTION_MAP[k] for k in sorted(EMOTION_MAP)]
LABEL_TO_ID = {name: i for i, name in enumerate(EMOTIONS)}
ID_TO_LABEL = {i: name for name, i in LABEL_TO_ID.items()}

SAMPLE_RATE = 16000
N_MFCC = 40
MAX_FRAMES = 180

print("TensorFlow:", tf.__version__)
print("Labels:", EMOTIONS)

## Step 4 — Build a file manifest
Only audio-only **speech** recordings are used. Actor IDs 01–16 are used for training, 17–20 for validation, and 21–24 for the final test set. The test set is therefore made of actors the model did not train on.

In [ ]:
records = []
for path in extract_dir.rglob("*.wav"):
    parts = path.stem.split("-")
    if len(parts) != 7:
        continue
    modality, vocal_channel, emotion_code, intensity, statement, repetition, actor = parts
    # Audio-only modality = 03; speech vocal channel = 01
    if modality != "03" or vocal_channel != "01":
        continue
    if emotion_code not in EMOTION_MAP:
        continue
    actor_id = int(actor)
    split = "train" if actor_id <= 16 else ("validation" if actor_id <= 20 else "test")
    records.append({
        "path": str(path),
        "emotion_code": emotion_code,
        "emotion": EMOTION_MAP[emotion_code],
        "actor": actor_id,
        "split": split,
    })

df = pd.DataFrame(records).drop_duplicates(subset="path").reset_index(drop=True)
print("Total speech files:", len(df))
display(df.head())
print("\nFiles by split:")
display(pd.crosstab(df["split"], df["emotion"]))
print("\nUnique actors per split:")
print(df.groupby("split")["actor"].nunique())

## Step 5 — Extract MFCC features
Each recording becomes a sequence of 40 MFCC features per audio frame. Sequences are padded or truncated to 180 frames so that they can be batched by the LSTM.

In [ ]:
def extract_mfcc(audio_path, sample_rate=SAMPLE_RATE, n_mfcc=N_MFCC, max_frames=MAX_FRAMES):
    y, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
    if y.size == 0:
        raise ValueError(f"Empty audio file: {audio_path}")
    # Trim leading/trailing silence; retain original if trimming removes everything
    trimmed, _ = librosa.effects.trim(y, top_db=30)
    if trimmed.size > 0:
        y = trimmed
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=n_mfcc, n_fft=400, hop_length=160
    ).T.astype(np.float32)  # shape: time frames x MFCC coefficients

    # Per-recording normalization helps reduce loudness/scale variation
    mean = mfcc.mean(axis=0, keepdims=True)
    std = mfcc.std(axis=0, keepdims=True) + 1e-6
    mfcc = (mfcc - mean) / std

    if mfcc.shape[0] < max_frames:
        pad = np.zeros((max_frames - mfcc.shape[0], n_mfcc), dtype=np.float32)
        mfcc = np.vstack([mfcc, pad])
    else:
        mfcc = mfcc[:max_frames]
    return mfcc

X_list, y_list, split_list, paths_list = [], [], [], []
errors = []

for row in df.itertuples(index=False):
    try:
        X_list.append(extract_mfcc(row.path))
        y_list.append(LABEL_TO_ID[row.emotion])
        split_list.append(row.split)
        paths_list.append(row.path)
    except Exception as exc:
        errors.append((row.path, str(exc)))

X = np.stack(X_list).astype(np.float32)
y = np.array(y_list, dtype=np.int64)
splits = np.array(split_list)
paths = np.array(paths_list)

print("Feature tensor shape:", X.shape)  # samples, time frames, MFCCs
print("Labels shape:", y.shape)
print("Failed files:", len(errors))
if errors[:3]:
    print("First errors:", errors[:3])

## Step 6 — Prepare train, validation and test sets

In [ ]:
train_mask = splits == "train"
val_mask = splits == "validation"
test_mask = splits == "test"

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Train classes:", np.bincount(y_train, minlength=len(EMOTIONS)))

## Step 7 — Build the LSTM model
The network reads MFCC frames as a sequence and learns temporal patterns. Dropout helps reduce overfitting.

In [ ]:
from tensorflow.keras import layers, models, callbacks

model = models.Sequential([
    layers.Input(shape=(MAX_FRAMES, N_MFCC)),
    layers.Masking(mask_value=0.0),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.35),
    layers.LSTM(64),
    layers.Dropout(0.35),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(len(EMOTIONS), activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## Step 8 — Train the model
Early stopping restores the best validation-loss weights. You can lower `EPOCHS` if you need a quicker demonstration.

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}

early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
)

EPOCHS = 40
BATCH_SIZE = 32

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)

## Step 9 — Plot training history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.history["accuracy"], label="Train accuracy")
axes[0].plot(history.history["val_accuracy"], label="Validation accuracy")
axes[0].set_title("Model accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train loss")
axes[1].plot(history.history["val_loss"], label="Validation loss")
axes[1].set_title("Model loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
plt.tight_layout()

history_path = RESULTS_DIR / "training_history.png"
plt.savefig(history_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", history_path)

## Step 10 — Evaluate on the held-out actors
These results are from the test actors only. Your exact metrics will depend on training, hardware, and preprocessing; do not assume a specific accuracy before running the notebook.

In [ ]:
test_probs = model.predict(X_test, batch_size=32, verbose=0)
test_pred = np.argmax(test_probs, axis=1)

test_accuracy = accuracy_score(y_test, test_pred)
print(f"Test accuracy: {test_accuracy:.4f}\n")
report = classification_report(
    y_test, test_pred, labels=list(range(len(EMOTIONS))),
    target_names=EMOTIONS, zero_division=0
)
print(report)

report_path = RESULTS_DIR / "classification_report.txt"
report_path.write_text(
    f"Test accuracy: {test_accuracy:.4f}\n\n{report}", encoding="utf-8"
)

cm = confusion_matrix(y_test, test_pred, labels=list(range(len(EMOTIONS))))
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=EMOTIONS, yticklabels=EMOTIONS
)
plt.xlabel("Predicted emotion")
plt.ylabel("True emotion")
plt.title("Confusion Matrix — Held-out Actors")
plt.tight_layout()
cm_path = RESULTS_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved report:", report_path)
print("Saved confusion matrix:", cm_path)

## Step 11 — Save the trained model and label configuration
Download these artifacts after training. Colab's local runtime storage is temporary and can be cleared when the runtime disconnects.

In [ ]:
MODEL_PATH = MODELS_DIR / "emotion_lstm.keras"
LABELS_PATH = MODELS_DIR / "emotion_labels.json"
CONFIG_PATH = MODELS_DIR / "model_config.json"

model.save(MODEL_PATH)
LABELS_PATH.write_text(json.dumps(ID_TO_LABEL, indent=2), encoding="utf-8")
CONFIG_PATH.write_text(json.dumps({
    "sample_rate": SAMPLE_RATE,
    "n_mfcc": N_MFCC,
    "max_frames": MAX_FRAMES,
    "emotions": EMOTIONS,
    "dataset": "RAVDESS Speech Audio",
    "split": "train actors 01-16; validation 17-20; test 21-24",
}, indent=2), encoding="utf-8")

print("Saved model:", MODEL_PATH)
print("Saved labels:", LABELS_PATH)
print("Saved config:", CONFIG_PATH)

## Step 12 — Predict an uploaded audio file
The model returns a predicted label and softmax scores for all eight classes. These scores are not calibrated probabilities and should not be interpreted as certainty about a person's real emotional state.

In [ ]:
def predict_emotion(audio_path):
    if audio_path is None:
        return "Please upload a WAV audio file.", {}, None

    features = extract_mfcc(audio_path)
    probabilities = model.predict(features[np.newaxis, ...], verbose=0)[0]
    best_idx = int(np.argmax(probabilities))
    label = ID_TO_LABEL[best_idx]
    score_dict = {ID_TO_LABEL[i]: float(probabilities[i]) for i in range(len(EMOTIONS))}

    # MFCC plot for this recording
    y_audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    mfcc_plot = librosa.feature.mfcc(
        y=y_audio, sr=sr, n_mfcc=N_MFCC, n_fft=400, hop_length=160
    )
    fig, ax = plt.subplots(figsize=(10, 4))
    img = librosa.display.specshow(mfcc_plot, x_axis="time", sr=sr, hop_length=160, ax=ax)
    ax.set_title("MFCC Feature Map")
    fig.colorbar(img, ax=ax, format="%+2.0f")
    fig.tight_layout()

    result = f"Predicted emotion: {label.upper()}\nTop model score: {probabilities[best_idx]:.1%}"
    return result, score_dict, fig

## Step 13 — Launch the interactive Gradio UI
Upload a WAV file, press **Predict emotion**, and view the predicted emotion, class scores, and MFCC visualization. You may upload a RAVDESS sample or your own recording.

In [ ]:
import gradio as gr
import librosa.display

with gr.Blocks(theme=gr.themes.Soft(), title="Speech Emotion Recognition") as demo:
    gr.Markdown("# 🎙️ Speech Emotion Recognition")
    gr.Markdown(
        "Upload a speech recording to estimate its expressed emotion using an LSTM trained on RAVDESS. "
        "This is an educational model; predictions can be wrong."
    )
    with gr.Row():
        audio_input = gr.Audio(
            label="Upload audio (WAV recommended)",
            type="filepath"
        )
        with gr.Column():
            result_output = gr.Textbox(label="Prediction", lines=2)
            scores_output = gr.Label(label="Model scores for each emotion", num_top_classes=8)
    mfcc_output = gr.Plot(label="MFCC visualization")
    predict_button = gr.Button("Predict emotion", variant="primary")
    predict_button.click(
        fn=predict_emotion,
        inputs=audio_input,
        outputs=[result_output, scores_output, mfcc_output]
    )
    gr.Markdown(
        "**Responsible use:** Vocal emotion classification is uncertain and culturally/contextually dependent. "
        "Do not use this output for hiring, grading, diagnosis, surveillance, or other high-impact decisions."
    )

demo.launch(share=True, debug=False)

## Step 14 — Download the model and reports
Run this cell after training and evaluation. It creates a ZIP containing the model and result files for saving outside the temporary Colab runtime.

In [ ]:
import shutil
from google.colab import files

bundle_path = Path("/content/Emotion_Recognition_LSTM_artifacts.zip")
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in [MODELS_DIR, RESULTS_DIR]:
        for file_path in folder.rglob("*"):
            if file_path.is_file():
                zf.write(file_path, arcname=str(file_path.relative_to(PROJECT_DIR)))

print("Created:", bundle_path)
files.download(str(bundle_path))

## Troubleshooting
- **Download error:** Open https://zenodo.org/records/11063852 and manually upload `Audio_Speech_Actors_01-24_16k.zip` into `/content/Emotion_Recognition_LSTM/data/`, then rerun extraction.
- **No WAV files found:** Check that the ZIP was extracted and that files have `.wav` extensions.
- **Out of memory:** Use a GPU runtime, lower `BATCH_SIZE` to 16, or lower `MAX_FRAMES` to 140 and rerun feature extraction and training.
- **Gradio link unavailable:** Rerun the UI cell; Colab share links are temporary. Do not upload private recordings to a public demo.
- **Unexpectedly high accuracy:** Keep actor-separated splitting as implemented; random file-level splitting can leak speaker characteristics across sets.
- **Generalization:** RAVDESS contains acted emotional speech, so real-world performance can differ substantially.